# Lesson13: LangChainの主要モジュール4【Retrieval（RAG）】

## 事前準備
シークレット機能でOpenAI APIのAPIキーを設定し、以下2つのコードを実行しましょう。

In [1]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

In [2]:
!pip install langchain==0.3.26 openai==1.91.0 langchain-community==0.3.26 httpx==0.28.1 chromadb==1.0.13

## 記述用

In [3]:
from langchain.chat_models import ChatOpenAI
from langchain.schema import SystemMessage,HumanMessage

llm = ChatOpenAI(model_name="gpt-4o-mini",temperature=0.5)

messages = [
    SystemMessage(content="You are a helpful assistant"),
    HumanMessage(content="日本の今の総理大臣は？")

]
result = llm(messages)
print(result.content)

/tmp/ipykernel_1189/2310800471.py:4: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  llm = ChatOpenAI(model_name="gpt-4o-mini",temperature=0.5)
/tmp/ipykernel_1189/2310800471.py:11: LangChainDeprecationWarning: The method `BaseChatModel.__call__` was deprecated in langchain-core 0.1.7 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result = llm(messages)


私のデータが2023年10月までのものであるため、その時点での情報をお伝えします。2023年10月時点での日本の総理大臣は岸田文雄（きしだ ふみお）氏です。彼は2021年10月4日に総理大臣に就任しました。最新の情報を確認する場合は、ニュースや公式な情報源を参照してください。


In [4]:
!pip install pymupdf==1.26.3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 68.2 MB/s eta 0:00:00


In [7]:
from langchain_community.document_loaders import PyMuPDFLoader
#LangChainでPDFファイルを高速に読み込むためのドキュメントローダー

loader = PyMuPDFLoader("healthX_instructions.pdf")
pages = loader.load()

len(pages)




12

In [8]:
pages[0]

Document(metadata={'producer': 'Skia/PDF m133 Google Docs Renderer', 'creator': '', 'creationdate': '', 'source': 'healthX_instructions.pdf', 'file_path': 'healthX_instructions.pdf', 'total_pages': 12, 'format': 'PDF 1.4', 'title': 'healthX_instructions', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0}, page_content='健康管理アプリ「HealthX」サービス詳細\n1. はじめに\n「HealthX」は、現代の多忙なライフスタイルに対応しつつ、個人の健康管理をサポートするため\nに設計された、革新的な健康管理アプリケーションです。現代社会において、仕事や家事、その\n他のライフスタイルの中で、健康を意識した行動を取り続けることは簡単ではありません。しか\nし、健康的な習慣を形成し維持することは、心身のバランスを整え、長期的な幸福を実現するた\nめに欠かせない要素です。「HealthX」は、こうした課題に対処するため、ユーザーにとって最適\nな健康管理のパートナーとなることを目指しています。\n1.1 健康管理の重要性と課題\n健康は、単に病気がない状態を意味するものではなく、身体的、精神的、社会的に完全に良好な\n状態にあることを指します。しかし、現代の生活環境では、ストレスや不規則な生活、運動不足、\n栄養の偏りなど、さまざまな要因が健康に悪影響を与えがちです。忙しい日常の中で健康を意識\nし、持続的にケアすることは、時間や知識、モチベーションが必要であり、多くの人々にとって難し\nい課題です。\n1.2 「HealthX」が提供する解決策\n「HealthX」は、このような現代の健康管理の課題に対応するため、AI技術とデータ解析を駆使し\nて、パーソナライズされた健康管理のアプローチを提供します。個々のユー

In [9]:
pages[0].page_content

'健康管理アプリ「HealthX」サービス詳細\n1. はじめに\n「HealthX」は、現代の多忙なライフスタイルに対応しつつ、個人の健康管理をサポートするため\nに設計された、革新的な健康管理アプリケーションです。現代社会において、仕事や家事、その\n他のライフスタイルの中で、健康を意識した行動を取り続けることは簡単ではありません。しか\nし、健康的な習慣を形成し維持することは、心身のバランスを整え、長期的な幸福を実現するた\nめに欠かせない要素です。「HealthX」は、こうした課題に対処するため、ユーザーにとって最適\nな健康管理のパートナーとなることを目指しています。\n1.1 健康管理の重要性と課題\n健康は、単に病気がない状態を意味するものではなく、身体的、精神的、社会的に完全に良好な\n状態にあることを指します。しかし、現代の生活環境では、ストレスや不規則な生活、運動不足、\n栄養の偏りなど、さまざまな要因が健康に悪影響を与えがちです。忙しい日常の中で健康を意識\nし、持続的にケアすることは、時間や知識、モチベーションが必要であり、多くの人々にとって難し\nい課題です。\n1.2 「HealthX」が提供する解決策\n「HealthX」は、このような現代の健康管理の課題に対応するため、AI技術とデータ解析を駆使し\nて、パーソナライズされた健康管理のアプローチを提供します。個々のユーザーのライフスタイ\nル、健康状態、目標に応じたカスタマイズされたアドバイスやサポートが得られるため、忙しい中\nでも無理なく健康的な生活習慣を維持できるよう支援します。さらに、アプリは操作が非常に簡単\nであり、日常の活動記録や健康データの管理が手軽に行えるため、健康に関する知識が少ない\n人でも効果的に活用できます。\n1.3 主要な機能\n「HealthX」は、栄養、運動、睡眠、メンタルヘルスといった健康の主要な要素をカバーしていま\nす。具体的には、毎日の食事記録を通じて栄養素のバランスをチェックしたり、運動計画を立て\nてフィットネスを促進することができます。さらに、ユーザーの睡眠パターンを追跡し、睡眠の質を\n向上させるための提案を行うことで、総合的な健康状態の改善を図ります。メンタルヘルスの面\nでは、ストレスレベルの測定やリラクゼーション技術の提案も行い、

In [10]:
len(pages[0].page_content)

1011

In [11]:
#LangChain が提供する OpenAI の埋め込みモデル（Embedding API）を利用するためのクラス
from langchain.embeddings.openai import OpenAIEmbeddings

#文章をコンピュータが理解・計算しやすい高次元のベクトルデータに変換
embeddings = OpenAIEmbeddings()

/tmp/ipykernel_1189/1834186319.py:5: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import OpenAIEmbeddings``.
  embeddings = OpenAIEmbeddings()


In [12]:
!pip install tiktoken==0.11.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 17.6 MB/s eta 0:00:00
  Attempting uninstall: tiktoken
    Found existing installation: tiktoken 0.13.0
    Uninstalling tiktoken-0.13.0:
      Successfully uninstalled tiktoken-0.13.0


In [13]:
vector_data = embeddings.embed_query("犬")
print(vector_data)

[-0.002103286730499773, -0.021205741258615435, -0.011823149964930125, -0.02770033765899977, -0.01795166303028901, 0.012534979344545895, -0.0019931226820192978, -0.007687758883098629, -0.014833172326132068, -0.014548440015492232, 0.002435473882974762, 0.03910317151441076, -0.006155630849589545, 0.004260808098138319, -0.009002948833713892, 0.01867027150671649, 0.026127533181975332, 0.003830320782264621, 0.02687325990829474, -0.012094323150623995, -0.012507862025976507, 0.0058946267747745145, 0.0008872446796306377, -0.014033210962673882, -0.011165555222453131, 0.0004838745950148377, 0.0225751653807082, -0.011124878778937778, 0.000489806508451891, -0.014263708636174943, 0.037096488077631044, -0.00033112786039050024, -0.02128709228300105, -0.007680979786286919, 0.001554161145885007, -0.04024209703167992, -0.0020439677707522167, -0.011484183017151517, 0.0044269020236217685, -0.008291119454098123, 0.0006953049236836111, 0.0139383008133423, 0.0016838157961910586, 0.004030310890298531, -0.00042

In [14]:
!pip install chromadb==1.0.13

In [ ]:
from langchain.chains import RetrievalQA
#LangChainでRAG（検索拡張生成）を手軽に実装するためのチェーン（Chain）クラスを読み込むためのコード
from langchain.chat_models import ChatOpenAI
#PythonのAI開発ライブラリ「LangChain」から、OpenAI社のチャット用AIモデル（ChatGPTなど）を呼び出すためのクラスを読み込むコード


llm = ChatOpenAI(model_name="gpt-4o-mini",temperature=0.5)
#AI開発ツールのLangChainを使って、OpenAIの軽量チャットモデルを準備する
chain = RetrievalQA.from_chain_type(llm=llm,chain_type="stuff",retriever=retriever)

#AI開発フレームワーク「LangChain」で、社内文書や外部データをもとに質問へ答える仕組み（RAG）を簡単に作る(llm=llm: 使う AIモデル,chain_type="stuff": 検索した文書を そのまままとめてAIに渡す)
#文書データから 質問に関係ある部分を探す機能

query = "HealthXの、ユーザー体験向上のための仕掛けを一言で教えてください。"

result = chain.run(query)
print(result)